## 2.1 Zero-Shot MMLU baseline

In [ ]:
from typing import Any
import re

def parse_mmlu_response(
    response: str,
    mmlu_example: dict[str, Any] | None = None,
):
    chars = re.findall(r'the correct answer is (.)', response.lower())
    for char in chars[::-1]:
        if char in ["a", "b", "c", "d"]:
            return char.upper()
    return None


In [ ]:
from vllm import LLM, SamplingParams
llm = LLM(model="meta-llama/Llama-3.1-8B")

# Create a sampling params object, stopping generation on newline.
sampling_params = SamplingParams(
    temperature=0.0, top_p=1.0, max_tokens=1024, stop=["# Query:"]
)


In [ ]:
from pathlib import Path
mmlu_eval_dir = Path("/home/azureuser/localfiles/cs336-assignment5-alignment-mine/data/mmlu/val")

import pandas as pd
mmlu_examples = []
for file in mmlu_eval_dir.glob("*.csv"):
    subject = file.name.split("_val.csv")[0]
    df = pd.read_csv(file, names=["question", "A", "B", "C", "D", "answer"])
    df["subject"] = subject
    df["options"] = df[["A", "B", "C", "D"]].values.tolist()
    mmlu_examples.extend(df.to_dict("records"))



In [ ]:
mmlu_prompt_file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/cs336_alignment/prompts/mmlu.prompt"

with open(mmlu_prompt_file) as f:
    mmlu_prompt_template = f.read()

# print(mmlu_prompt_template)
mmlu_instructions = [
    mmlu_prompt_template.format(
        subject = example["subject"],
        question = example["question"],
        options = example["options"]
    ) for example in mmlu_examples
]
# print(mmlu_instructions[0])

In [ ]:
zero_shot_prompt_file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/cs336_alignment/prompts/zero_shot_system_prompt.prompt"

with open(zero_shot_prompt_file) as f:
    zero_shot_prompt_template = f.read()

# print(zero_shot_prompt_template)
prompts = [zero_shot_prompt_template.format(instruction=instruction) for instruction in mmlu_instructions]

In [ ]:
outputs = llm.generate(prompts[:5], sampling_params, use_tqdm=False)
outputs = [opt.outputs[0].text for opt in outputs]

In [ ]:
answers = [parse_mmlu_response(opt) for opt in outputs]
ground_truths = [eg["answer"] for eg in mmlu_examples[:5]]

In [ ]:
sum([ans == gt for ans, gt in zip(answers, ground_truths)]) / len(answers)

In [ ]:
print(mmlu_instructions[0])

In [ ]:
from vllm import LLM, SamplingParams
llm = LLM(model="Qwen/Qwen2.5-Math-1.5B")

# Sample prompts.
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]

# Create a sampling params object, stopping generation on newline.
sampling_params = SamplingParams(
    temperature=1.0, top_p=1.0, max_tokens=1024, stop=["\n"]
)

# Generate texts from the prompts. The output is a list of RequestOutput objects
# that contain the prompt, generated text, and other information.
outputs = llm.generate(prompts, sampling_params)

In [ ]:
# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")

In [ ]:
from vllm import LLM, SamplingParams
from typing import Callable, List
from cs336_alignment.drgrpo_grader import r1_zero_reward_fn
import json
import pandas as pd
from os import PathLike

def get_prompts(prompt_template, problems):
    prompts = [prompt_template.replace("{question}", p) for p in problems]
    return prompts

def evaluate_vllm(
    vllm_model: LLM,
    eval_sampling_params: SamplingParams,
    prompts: List[str],
    solutions: List[str],
    reward_fn: Callable[[str, str], dict[str, float]],
    output_file: str | PathLike | None = None
) -> tuple[list, list]:
    """
    Evaluatea languagemodelon a listof prompts,
    compute evaluation metrics, and serialize results to disk.
    """
    responses = vllm_model.generate(prompts, eval_sampling_params, use_tqdm=False)
    solutions_generated = [opt.outputs[0].text for opt in responses]

    evals = [reward_fn(sol_gen, sol) for sol_gen, sol in zip(solutions_generated, solutions)]

    # Serialize the prompts, solutions, solutions generated, and corresponding evals to disk
    if output_file:
        with open(output_file, 'w') as f:
            for prompt, solution, sol_gen, eval_dict in zip(prompts, solutions, solutions_generated, evals):
                result = {
                    "prompt": prompt,
                    "ground_truth": solution,
                    "generated": sol_gen,
                    "eval": eval_dict
                }
                f.write(json.dumps(result) + '\n')

    return evals, solutions_generated

In [ ]:
# llm = LLM(model="Qwen/Qwen2.5-Math-1.5B")
llm = LLM(model="./sft_model/")

prompt_r1_zero_file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/cs336_alignment/prompts/r1_zero.prompt"
with open(prompt_r1_zero_file) as f:
    prompt_r1_zero = f.read()

df = pd.read_json("/home/azureuser/localfiles/cs336-assignment5-alignment-mine/data/validation.jsonl", lines=True)
# df.head()

In [ ]:
# Create a sampling params object, stopping generation on newline.
sampling_params = SamplingParams(
    temperature=1.0, top_p=1.0, max_tokens=1024, stop=["</answer>"],
    include_stop_str_in_output=True,
)

prompts = get_prompts(prompt_r1_zero, df.problem.tolist())
evals, answers_generated = evaluate_vllm(
    llm, sampling_params, prompts, df.answer.tolist(), r1_zero_reward_fn,
    output_file="eval_results_sft.jsonl"
)

## 3.1 Using vLLM for offline language model inference
### math_baseline
1. Done
1. Commentary on model and reward func perf
    1. See cell below for distribution. 
    1. For cases with zero format reward: most are because of the model failed to generate the answer tags or not in the right format. For cases with non-zero format reward but zero answer reward: 50/50 of wrong answer and parser failure
1. less than 3% get both format and answer rewards

In [ ]:
# math_baseline.2
df_eval = pd.DataFrame(evals)
# Check rows where format_reward is 1 and answer_reward is 1
print(f"Format reward 1, Answer reward 1: {((df_eval["format_reward"] == 1) & (df_eval["answer_reward"] == 1)).sum()}")
print(f"Format reward 1, Answer reward 0: {((df_eval["format_reward"] == 1) & (df_eval["answer_reward"] == 0)).sum()}")
print(f"Format reward 0, Answer reward 0: {((df_eval["format_reward"] == 0) & (df_eval["answer_reward"] == 0)).sum()}")

In [ ]:
counts_by_category = df.groupby("subject")["problem"].count()
accurate_by_category = df[df_eval.reward == 1].groupby("subject")["problem"].count()
accuracy_percentage = (accurate_by_category / counts_by_category) * 100
accuracy_percentage

In [ ]:
sample_ids = df_eval[(df_eval.format_reward==1) & (df_eval.answer_reward==0)].sample(10).index
sample_problems = (df["problem"].tolist()[i] for i in sample_ids)
sample_answers = (df["answer"].tolist()[i] for i in sample_ids)
sample_answers_generated = (answers_generated[i] for i in sample_ids)

In [ ]:
print(next(sample_problems))
print()
print(next(sample_answers))
print()
print(next(sample_answers_generated))


## 4. Supervised Finetuning for MATH

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Math-1.5B",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Math-1.5B")

### 4.2 SFT helper methods

In [ ]:
import random
from transformers import PreTrainedModel, PreTrainedTokenizer
from sft_helper_methods import tokenize_prompt_and_output, compute_entropy

def log_generations(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    step: int,
    prompts: list[str],
    responses: list[str],
    ground_truths: list[str],
    evals: list[dict],
):
    log = {"step": step}
    # evaluation on full dataset
    df = pd.DataFrame(evals)
    ids_format = df[df.format_reward == 1].index.tolist()
    ids_answer = df[df.answer_reward == 1].index.tolist()
    ids_total = df[df.reward == 1].index.tolist()
    log["format_reward"] = round(len(ids_format) / len(df), 3)
    log["answer_reward"] = round(len(ids_answer) / len(df), 3)
    log["reward"] = round(len(ids_total) / len(df), 3)

    res_len = sum([len(res.split()) for res in responses])
    res_len_correct = sum([len(responses[i].split()) for i in ids_total])
    avg_res_len_correct = res_len_correct / len(ids_total)
    avg_res_len_incorrect = (res_len - res_len_correct) / (len(responses) - len(ids_total))
    log["average_response_length"] = round(res_len / len(responses), 3)
    log["average_response_length_correct"] = round(avg_res_len_correct, 3)
    log["average_response_length_incorrect"] = round(avg_res_len_incorrect, 3)

    # small sample for token entropy
    sample_logs = []
    sample_ids = random.sample(range(len(prompts)), 2)
    samples_tokenized = tokenize_prompt_and_output(
        [prompts[i] for i in sample_ids],
        [responses[i] for i in sample_ids],
        tokenizer,
    )
    input_ids = samples_tokenized["input_ids"].to(model.device)
    with torch.inference_mode():
        logits = model(input_ids).logits
        entropies = compute_entropy(logits)
    
    for i, id in enumerate(sorted(sample_ids)):
        sample_log = {
            "id": id, 
            "prompt": prompts[id],
            "ground_truth": ground_truths[id],
            "response": responses[id],
            "response_average_token_entropy": torch.mean(entropies, -1)[i].item(),
            "eval": evals[id]
        }
        sample_logs.append(sample_log)
    
    log["samples"] = sample_logs
    return log

        

In [ ]:
log = log_generations(
    model, tokenizer, 11, prompts, responses, ground_truths, evals
)

import json
print(json.dumps(log, indent=2))

In [ ]:
from nltk.tokenize import word_tokenize

text = "\nGiven the quadratic equation \\(x^2 + 5x + 7 = 0\\) with roots \\(r\\), we need to compute \\((r - 1)(r + 2)(r + 6)(r + 3)\\).\n\nFirst, rewrite the expression:\n\\[\n(r - 1)(r + 2)(r + 6)(r + 3) = [(r - 1)(r + 3)][(r + 2)(r + 6)].\n\\]\n\nCalculate each pair separately:\n\\[\n(r - 1)(r + 3) = r^2 + 2r - 3,\n\\]\n\\[\n(r + 2)(r + 6) = r^2 + 8r + 12.\n\\]\n\nMultiply the results:\n\\[\n(r^2 + 2r - 3)(r^2 + 8r + 12).\n\\]\n\nExpand using FOIL:\n\\[\nr^4 + 8r^3 + 12r^2 + 2r^3 + 16r^2 + 24r - 3r^2 - 24r - 36.\n\\]\n\nCombine like terms:\n\\[\nr^4 + 10r^3 + 25r^2 - 36.\n\\]\n\nSince \\(r\\) is a root of \\(x^2 + 5x + 7 = 0\\), substitute \\(r^2 = -5r - 7\\):\n\\[\nr^4 = (-5r - 7)^2 = 25r^2 + 70r + 49,\n\\]\n\\[\n10r^3 = 10r(-5r - 7) = -50r^2 - 70r,\n\\]\n\\[\n25r^2 = 25r^2,\n\\]\n\\[\n-36 = -36.\n\\]\n\nCombine all terms:\n\\[\n(25r^2 + 70r + 49) + (-50r^2 - 70r) + 25r^2 - 36 = 49 - 36 = 13.\n\\]\n\nThus, the final answer is:\n\\[\n\\boxed{13}.\n\\]\n</think>\n<answer>13</answer>"
tokens = word_tokenize(text)

### 4.3 SFT Experiment

In [ ]:
# filtering: SFT examples that produce the correct answer.
from vllm import LLM, SamplingParams
from cs336_alignment.drgrpo_grader import r1_zero_reward_fn
import json
import pandas as pd
from os import PathLike

vllm_model = LLM(model="Qwen/Qwen2.5-Math-1.5B")

df = pd.read_json("/home/azureuser/localfiles/cs336-assignment5-alignment-mine/data/sft.jsonl", lines=True).drop_duplicates().reset_index(drop=True)
# df.head()

# Create a sampling params object, stopping generation on newline.
sampling_params = SamplingParams(
    temperature=1.0, top_p=1.0, max_tokens=1024, stop=["</answer>"],
    include_stop_str_in_output=True,
)

responses = vllm_model.generate(df.prompt.tolist(), sampling_params, use_tqdm=False)
solutions_generated = [opt.outputs[0].text for opt in responses]

evals = [r1_zero_reward_fn(sol_gen, sol) for sol_gen, sol in zip(solutions_generated, df.ground_truth.tolist())]
df = pd.DataFrame(evals)
df[df.reward==1].index.tolist()

In [ ]:
# compare before and after `SFT`
import pandas as pd
import json
import random

eval_res = pd.read_json("eval_results.jsonl", lines=True).to_dict(orient="records")
eval_res_sft = pd.read_json("eval_results_sft.jsonl", lines=True).to_dict(orient="records")

sample_ids = random.sample(range(len(eval_res)), 10)

def b4_aft_sft(sample_ids):
    for id in sample_ids:
        print(id)
        print(json.dumps(eval_res[id], indent=2))
        print("="*100)
        print(json.dumps(eval_res_sft[id], indent=2))
        yield

gen = b4_aft_sft(sample_ids)

## 5. Expert Iteration for MATH

In [ ]:
from vllm import LLM, SamplingParams
from typing import Callable, List
from cs336_alignment.drgrpo_grader import r1_zero_reward_fn
import json
import pandas as pd
from os import PathLike

def sample_expert_outputs(
    vllm_model: LLM,
    eval_sampling_params: SamplingParams,
    prompt_template: str,
    problems: List[str],
    answers: List[str],
    reward_fn: Callable[[str, str], dict[str, float]],
    use_tqdm: bool = False
) -> tuple[list, list]:
    """
    Sample G outputs for each question, then select `expert` ones with `reward==1`.
    Questions can have mulitple or zero valid outputs; increase G if num of valid is too low.
    """
    prompts = [prompt_template.replace("{question}", p) for p in problems]

    responses = vllm_model.generate(prompts, eval_sampling_params, use_tqdm=use_tqdm)
    answers_generated = [[output.text for output in response.outputs] for response in responses]
    evals = [[reward_fn(ans_gen, ans) for ans_gen in ans_gens] for ans_gens, ans in zip(answers_generated, answers)]

    answers_generated = [a for ans_gen in answers_generated for a in ans_gen]
    evals = [e for eval in evals for e in eval]
    ids2keep = [i for i in range(len(evals)) if evals[i]["reward"] == 1]

    n_gen_per_prompt = eval_sampling_params.n  # one prompt maps to multiple generated
    return [
        {
            "prompt": prompts[i//n_gen_per_prompt], 
            "response": answers_generated[i],
            "ground_truth": answers[i//n_gen_per_prompt],
        } for i in ids2keep
    ]

In [ ]:
llm = LLM(model="Qwen/Qwen2.5-Math-1.5B")

prompt_r1_zero_file = "/home/azureuser/localfiles/cs336-assignment5-alignment-mine/cs336_alignment/prompts/r1_zero.prompt"
with open(prompt_r1_zero_file) as f:
    prompt_r1_zero = f.read()


In [ ]:
df = pd.read_json("/home/azureuser/localfiles/cs336-assignment5-alignment-mine/data/train.jsonl", lines=True)
print(df.shape)
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)
# df = df.iloc[:2]


In [ ]:
df = df.sample(64)

# Create a sampling params object, stopping generation on newline.
G = 4
sampling_params = SamplingParams(
    temperature=1.0, top_p=1.0, stop=["</answer>"],
    include_stop_str_in_output=True,
    max_tokens=1024, 
    min_tokens=4, 
    n=G
)

# prompts = get_prompts(prompt_r1_zero, df.problem.tolist())

# responses = llm.generate(prompts, sampling_params, use_tqdm=True)
# solutions_generated = [output.text for response in responses for output in response.outputs]
# evals = [reward_fn(sol_gen, sol) for sol_gens, sol in zip(solutions_generated, solutions) for sol_gen in sol_gens]
sample_expert_outputs = sample_expert_outputs(
    llm, sampling_params, prompt_r1_zero, df.problem.tolist(), df.answer.tolist(), r1_zero_reward_fn, use_tqdm=True,
)

In [ ]:
sample_expert_outputs[2]

In [ ]:
dd = pd.DataFrame(evals)
dd[dd.reward==1]

## 7 GRPO
### 7.2 Implementation

In [ ]:
from typing import Callable
import torch
from cs336_alignment.drgrpo_grader import r1_zero_reward_fn

def compute_group_normalized_rewards(
    reward_fn: Callable,
    rollout_respones: list[str],
    repeated_ground_truths: list[str],
    group_size: int,
    advantage_eps: float,
    normalize_by_std: bool,
) -> tuple[torch.Tensor, torch.Tensor, dict[str, float]]:
    # use `group_size` to calculate by group
    raw_rewards = torch.tensor(
        [reward_fn(ro, gt)["reward"] for ro, gt in zip(rollout_respones, repeated_ground_truths)]
    ).view(-1, group_size)

    advantages = raw_rewards - raw_rewards.mean(dim=-1, keepdim=True)

    if normalize_by_std:
        advantages = advantages / (raw_rewards.std(dim=-1, keepdim=True) + advantage_eps)
    return advantages.flatten(), raw_rewards.flatten()

In [ ]:
import torch

def compute_naive_policy_gradient_loss(
    raw_rewards_or_advantages: torch.Tensor,
    policy_log_probs: torch.Tensor,
) -> torch.Tensor:  
    """advantage is on rollout level, therefore the same for every token in same rollout"""
    # batch_sz, seq_len = policy_log_probs.shape
    return -raw_rewards_or_advantages * policy_log_probs



In [ ]:
import torch

def compute_grpo_clip_loss(
    advantages: torch.Tensor,
    policy_log_probs: torch.Tensor,
    old_log_probs: torch.Tensor,
    cliprange: float,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    probs_ratio = torch.exp(policy_log_probs - old_log_probs)
    min_lhs = probs_ratio * advantages.unsqueeze(-1)
    min_rhs = torch.clip(probs_ratio, 1-cliprange, 1+cliprange) * advantages.unsqueeze(-1)
    loss = -1 * torch.min(min_lhs, min_rhs)

    # value mismatch when clipping happened
    clipped = min_lhs!=min_rhs
    metadata = {
        "clipped": clipped
    }

    return loss, metadata



In [ ]:
from typing import Literal

def compute_policy_gradient_loss(
    policy_log_probs: torch.Tensor,
    loss_type: Literal["no_baseline", "reinforce_with_baseline", "grpo_clip"],
    raw_rewards: torch.Tensor | None = None,
    advantages: torch.Tensor | None = None,
    old_log_probs: torch.Tensor | None = None,
    cliprange: float | None = None,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    if loss_type == "no_baseline":
        return raw_rewards, None
    elif loss_type == "reinforce_with_baseline":
        loss = compute_naive_policy_gradient_loss(
            advantages, policy_log_probs
        )
        return loss, None
    elif loss_type == "grpo_clip":
        loss, metadata = compute_grpo_clip_loss(
            advantages, policy_log_probs, old_log_probs, cliprange
        )
        return loss, metadata
    else:
        raise ValueError("Wrong loss type.")

In [ ]:
def masked_mean(
    tensor: torch.Tensor,
    mask: torch.Tensor,
    dim: int | None = None
) -> torch.Tensor:
    return torch.mean(tensor*mask, dim=dim)

In [ ]:
from typing import Literal

def grpo_microbatch_train_step(
    policy_log_probs: torch.Tensor,
    response_mask: torch.Tensor,
    gradient_accumulation_steps: int,
    loss_type: Literal["no_baseline", "reinforce_with_baseline", "grpo_clip"],
    raw_rewards: torch.Tensor | None = None,
    advantages: torch.Tensor | None = None,
    old_log_probs: torch.Tensor | None = None,
    cliprange: float | None = None,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    loss, metadata = compute_policy_gradient_loss(
        policy_log_probs, 
        loss_type,
        raw_rewards,
        advantages,
        old_log_probs,
        cliprange
    )
    loss = masked_mean(loss, response_mask, dim=-1) / gradient_accumulation_steps
    metadata["loss_type"] = loss_type
    return loss, metadata